# 07 — Graph Representation & Embeddings

## Objective
Test whether graph structure adds information through a scalable transaction×entity embedding based on sparse SVD.

> **Scientific contract:** fraud labels are validation ground truth, never unsupervised predictors. Chronological splits protect against future leakage. The test period is locked until Notebook 11.

### Outputs
Reproducible artifacts are saved to `artifacts/` and audit evidence to `reports/`. Interactive controls are for investigation/exploration; the underlying tables remain reproducible.

## 1. Create graph embeddings

In [1]:
import joblib
from pathlib import Path
import sys, json, warnings, numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets
warnings.filterwarnings("ignore")
ROOT=Path.cwd()
if not (ROOT/"data").exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/"src"))
from pipeline_utils import *
ART=ROOT/"artifacts"; REP=ROOT/"reports"; ART.mkdir(exist_ok=True); REP.mkdir(exist_ok=True)
RANDOM_STATE=42
fraud,_=load_data(); train,val,test,_=chronological_split(fraud); Ev,svd,Etr=graph_incidence_embedding(train,val,n_components=12); display(Markdown(f"**Embedding dimensions:** {Ev.shape[1]}")); display(pd.DataFrame({"component":range(1,len(svd.explained_variance_ratio_)+1),"variance":svd.explained_variance_ratio_}))

**Embedding dimensions:** 12

,component,variance
0,1,0.000122
1,2,0.000120
2,3,0.000119
3,4,0.000116
4,5,0.000115
5,6,0.000114
6,7,0.000112
7,8,0.000110
8,9,0.000109
9,10,0.000108


## 2. Inspect embedding geometry

In [2]:
px.line(pd.DataFrame({"component":range(1,len(svd.explained_variance_ratio_)+1),
                      "cumulative":np.cumsum(svd.explained_variance_ratio_)}),
                      x="component",
                      y="cumulative",
                      markers=True,
                      title="Cumulative explained variance").show();
px.scatter(pd.DataFrame(Ev[:10000,:2],columns=["z1","z2"]),x="z1",y="z2",title="Validation graph embedding space").show()

## 3. Graph novelty score

In [3]:
gscore=np.linalg.norm(Ev-Etr.mean(axis=0),axis=1);
print("Graph PR-AUC:",ranking_metrics(val['class'],gscore)['pr_auc']);
np.save(ART/"graph_train_embedding.npy",Etr);
np.save(ART/"graph_validation_embedding.npy",Ev); 
joblib.dump(svd,ART/"graph_svd.joblib")

Graph PR-AUC: 0.06614019381523806


['D:\\fraud_ecommerce_graph_anomaly_and_ml\\artifacts\\graph_svd.joblib']